# Melanoma & Nevus Dermoscopy Dataset - Image Pipeline

##### 1. Cấu hình môi trường (local / Colab)

Notebook này viết để chạy được cả ở local (test logic, CPU) lẫn trên Google Colab (train thật, GPU). Tự detect môi trường và set `DATA_ROOT` phù hợp — ở Colab cần mount Google Drive, nơi đã upload sẵn `images/` + `data/processed/`.

In [1]:
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_ROOT = '/content/drive/MyDrive/Melanoma'  # nơi chứa images.zip và data/processed
    PROCESSED_DIR = os.path.join(DRIVE_ROOT, 'data', 'processed')

    # Giải nén ra ổ đĩa cục bộ của Colab (/content) - nhanh hơn nhiều so với ghi thẳng lên Drive
    LOCAL_IMAGES_DIR = '/content/images'
    if not os.path.exists(LOCAL_IMAGES_DIR):
        zip_path = os.path.join(DRIVE_ROOT, 'images.zip')
        print(f'Giải nén {zip_path} ra {LOCAL_IMAGES_DIR} ...')
        os.system(f'unzip -q "{zip_path}" -d /content/')
        print('Xong.')

    IMAGES_DIR = LOCAL_IMAGES_DIR
else:
    PROCESSED_DIR = os.path.join('.', 'data', 'processed')
    IMAGES_DIR = os.path.join('.', 'images')

print('IN_COLAB:', IN_COLAB)
print('IMAGES_DIR:', IMAGES_DIR)
print('PROCESSED_DIR:', PROCESSED_DIR)

IN_COLAB: False
IMAGES_DIR: .\images
PROCESSED_DIR: .\data\processed


##### 2. Load metadata đã xử lý + join đường dẫn ảnh

In [2]:
import pandas as pd

df_train = pd.read_csv(os.path.join(PROCESSED_DIR, 'train.csv'))
df_val = pd.read_csv(os.path.join(PROCESSED_DIR, 'val.csv'))
df_test = pd.read_csv(os.path.join(PROCESSED_DIR, 'test.csv'))

for d in [df_train, df_val, df_test]:
    d['image_path'] = d['isic_id'].apply(lambda x: os.path.join(IMAGES_DIR, f'{x}.jpg'))

print('Train:', df_train.shape, '| Val:', df_val.shape, '| Test:', df_test.shape)

# Kiểm tra vài file ảnh đầu tiên có tồn tại thật không
missing_files = [p for p in df_train['image_path'].head(20) if not os.path.exists(p)]
print('Số file không tồn tại (trong 20 mẫu đầu):', len(missing_files))

Train: (12696, 25) | Val: (2752, 25) | Test: (2685, 25)
Số file không tồn tại (trong 20 mẫu đầu): 0


##### 3. Dataset class (PyTorch)

Trả về `(image_tensor, label, metadata_tensor)`. `metadata_tensor` gồm các feature đã engineer ở `03_imputation.ipynb` (`U50`, `U60`, `U70`, `sex_encoded`, 8 cột `anatom_*`, `age_approx_scaled`) — dùng nếu đi hướng multimodal, bỏ qua nếu đi hướng image-only.

In [3]:
import torch
from torch.utils.data import Dataset
from PIL import Image

METADATA_COLS = [
    'U50', 'U60', 'U70', 'sex_encoded', 'age_approx_scaled',
    'anatom_anterior torso', 'anatom_head/neck', 'anatom_lateral torso',
    'anatom_lower extremity', 'anatom_oral/genital', 'anatom_palms/soles',
    'anatom_posterior torso', 'anatom_upper extremity',
]
LABEL_MAP = {'nevus': 0, 'melanoma': 1}


class MelanomaDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row['image_path']).convert('RGB')
        if self.transform:
            image = self.transform(image)

        label = LABEL_MAP[row['diagnosis']]
        metadata = torch.tensor(row[METADATA_COLS].values.astype('float32'))

        return image, label, metadata

##### 4. Transform / augmentation

Augmentation (flip, rotation, color jitter) chỉ áp dụng cho `train`; `val`/`test` chỉ resize + normalize — không augment, để đánh giá phản ánh đúng dữ liệu thật.

In [4]:
from torchvision import transforms

IMAGE_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

##### 5. DataLoader

In [5]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32
NUM_WORKERS = 2 if IN_COLAB else 0

train_dataset = MelanomaDataset(df_train, transform=train_transform)
val_dataset = MelanomaDataset(df_val, transform=eval_transform)
test_dataset = MelanomaDataset(df_test, transform=eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print('Số batch - train:', len(train_loader), '| val:', len(val_loader), '| test:', len(test_loader))

Số batch - train: 397 | val: 86 | test: 84


##### 6. Sanity check — load thử 1 batch

In [6]:
images, labels, metadata = next(iter(train_loader))

print('images shape :', images.shape)   # (batch, 3, 224, 224)
print('labels shape :', labels.shape, '| vài label đầu:', labels[:8].tolist())
print('metadata shape:', metadata.shape)  # (batch, so metadata cols)

images shape : torch.Size([32, 3, 224, 224])
labels shape : torch.Size([32]) | vài label đầu: [1, 0, 0, 0, 0, 0, 1, 0]
metadata shape: torch.Size([32, 13])
